Authentication

High level overview:    
&emsp;&emsp;Use API key to generate a refresh token\
&emsp;&emsp;Use refresh token to generate an access token\
&emsp;&emsp;Use access token to make calls to Tradestation's servers

Refresh tokens last forever by default
Access tokens last for 20 minutes

Quickly mention: can revoke all refresh tokens using https://signin.tradestation.com/oauth/revoke endpoint

In [3]:
%load_ext autoreload
%autoreload 2

import json
import requests
import os
import dotenv

# Get the tradestation API key and secret using dotenv
dotenv.load_dotenv()

# Get the tradestation API key and secret
CLIENT_ID = os.getenv("TRADESTATION_API_KEY")
CLIENT_SECRET = os.getenv("TRADESTATION_SECRET")

print(CLIENT_ID)
print(CLIENT_SECRET)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
7sjAjleUcxRiYjpf4M9LBF3uDbN4Uumx
TasxAwowBUB58Q4Y0ypUluvmjJx6mSYkSIXFi_gNCefC2wFBPOWYKmitb8zyyAhZ


In [54]:
# generate a refresh token
# run this code block and then copy/paste the login URL into your brower and login with your TradeStation credentials
print(f'https://signin.tradestation.com/authorize?response_type=code&client_id={CLIENT_ID}&audience=https%3A%2F%2Fapi.tradestation.com&redirect_uri=http%3A%2F%2Flocalhost%3A3000&scope=openid%20MarketData%20profile%20ReadAccount%20Trade%20offline_access%20Matrix%20OptionSpreads')

https://signin.tradestation.com/authorize?response_type=code&client_id=7sjAjleUcxRiYjpf4M9LBF3uDbN4Uumx&audience=https%3A%2F%2Fapi.tradestation.com&redirect_uri=http%3A%2F%2Flocalhost%3A3000&scope=openid%20MarketData%20profile%20ReadAccount%20Trade%20offline_access%20Matrix%20OptionSpreads


In [4]:
# when you login, you will get a "code" returned in the URL
# paste the "code" into this variable assignment statement and run this block
CODE = 'ydHu8AzV6SUjzd5YNVlZNWBFO8Q48Dn56Uo1Fj_UvrreE'

# this request will get a new access token and refresh token
# if desired, you can paste the refresh token above and rerun that code block
# then after that, you can simply run the next code block anytime you need a new access token
url = "https://signin.tradestation.com/oauth/token"

payload=f'grant_type=authorization_code&client_id={CLIENT_ID}&client_secret={CLIENT_SECRET}&code={CODE}&redirect_uri=http%3A%2F%2Flocalhost%3A3000'
headers = {
  'Content-Type': 'application/x-www-form-urlencoded'
}

response = requests.request("POST", url, headers=headers, data=payload)
response_data = response.json()
REFRESH_TOKEN = response_data['refresh_token']
print('refresh_token: ', REFRESH_TOKEN)

KeyError: 'refresh_token'

In [5]:
REFRESH_TOKEN = os.environ.get('REFRESH_TOKEN') # your refresh token

In [6]:
# this step will get a new access token using your refresh token when this function is called
def get_access_token():
    url = "https://signin.tradestation.com/oauth/token"

    payload=f'grant_type=refresh_token&client_id={CLIENT_ID}&client_secret={CLIENT_SECRET}&refresh_token={REFRESH_TOKEN}'
    headers = {
      'Content-Type': 'application/x-www-form-urlencoded'
    }

    response = requests.request("POST", url, headers=headers, data=payload)
    response_data = response.json()
    print(f"response_data: {response_data}")
    # If response data contains an "error" key, raise an exception
    if 'error' in response_data:
        raise Exception(f"Error: {response_data['error']}")
    return response_data['access_token']

Simulation vs live connection

To use sim, you use sim-api. To use live, you just use api.

For example, for retrieving account information, the urls would look like the following:\
&emsp;&emsp;SIM: https://sim-api.tradestation.com/v3/brokerage/accounts \
&emsp;&emsp;Live: https://api.tradestation.com/v3/brokerage/accounts

In [7]:
core_url = "https://api.tradestation.com"

In [8]:
# get market data - snapshot
access_token = get_access_token()
print(f"Access token: {access_token}")

url = f"{core_url}/v3/marketdata/barcharts/@ES"

headers = {"Authorization": f'Bearer {access_token}'}

params = {
    "unit": "Minute",
    "interval": "5",
    "barsback": "10"
}

response = requests.request("GET", url, headers=headers, params=params)

response.json()

response_data: {'access_token': 'eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCIsImtpZCI6Ik56WXpOekExUXpORVJFTXpNMFZHTkVSRVFqSTBSakV4TmpZek56aEdRVFJETmtJd1JVVXpNUSJ9.eyJodHRwOi8vdHJhZGVzdGF0aW9uLmNvbS9pZ25vcmVfZHVhbF9sb2dvbiI6ImZhbHNlIiwiaHR0cDovL3RyYWRlc3RhdGlvbi5jb20vY2xpZW50X3RhZyI6IlEyaXQ4IiwiaHR0cDovL3RyYWRlc3RhdGlvbi5jb20vdXNlcm5hbWUiOiJrbXVycmF5MzAiLCJodHRwOi8vdHJhZGVzdGF0aW9uLmNvbS9mZGNuX2lkIjoiMTI1MTU3ODQiLCJodHRwOi8vdHJhZGVzdGF0aW9uLmNvbS9vbnl4X2lkIjo0NzI1MTMxLCJodHRwOi8vdHJhZGVzdGF0aW9uLmNvbS90ZW1wX2NyZWRlbnRpYWxzIjpmYWxzZSwiaXNzIjoiaHR0cHM6Ly9zaWduaW4udHJhZGVzdGF0aW9uLmNvbS8iLCJzdWIiOiJhdXRoMHwxMjUxNTc4NCIsImF1ZCI6WyJodHRwczovL2FwaS50cmFkZXN0YXRpb24uY29tIiwiaHR0cHM6Ly90cmFkZXN0YXRpb24tcHJvZC50c2xvZ2luLmF1dGgwLmNvbS91c2VyaW5mbyJdLCJpYXQiOjE3NjgxNjYwMDcsImV4cCI6MTc2ODE2NzIwNywic2NvcGUiOiJvcGVuaWQgcHJvZmlsZSBNYXJrZXREYXRhIFJlYWRBY2NvdW50IFRyYWRlIE1hdHJpeCBPcHRpb25TcHJlYWRzIG9mZmxpbmVfYWNjZXNzIiwiYXpwIjoiN3NqQWpsZVVjeFJpWWpwZjRNOUxCRjN1RGJONFV1bXgifQ.nJ-GuGde6Hs9-ls4hkCqtsYR4aHvTl4vh2v2y97

{'Bars': [{'High': '7004.25',
   'Low': '7002.75',
   'Open': '7004',
   'Close': '7004',
   'TimeStamp': '2026-01-09T21:15:00Z',
   'TotalVolume': '5630',
   'DownTicks': 1134,
   'DownVolume': 2809,
   'OpenInterest': '0',
   'IsRealtime': False,
   'IsEndOfHistory': False,
   'TotalTicks': 2508,
   'UnchangedTicks': 0,
   'UnchangedVolume': 0,
   'UpTicks': 1374,
   'UpVolume': 2821,
   'Epoch': 1767993300000,
   'BarStatus': 'Closed'},
  {'High': '7004.5',
   'Low': '7003.25',
   'Open': '7003.75',
   'Close': '7003.75',
   'TimeStamp': '2026-01-09T21:20:00Z',
   'TotalVolume': '1980',
   'DownTicks': 439,
   'DownVolume': 977,
   'OpenInterest': '0',
   'IsRealtime': False,
   'IsEndOfHistory': False,
   'TotalTicks': 974,
   'UnchangedTicks': 0,
   'UnchangedVolume': 0,
   'UpTicks': 535,
   'UpVolume': 1003,
   'Epoch': 1767993600000,
   'BarStatus': 'Closed'},
  {'High': '7004',
   'Low': '7002.75',
   'Open': '7003.75',
   'Close': '7003.25',
   'TimeStamp': '2026-01-09T21:25:

In [40]:
# get market data - streaming
access_token = get_access_token()

url = f"{core_url}/v3/marketdata/stream/barcharts/@ES"

headers = {"Authorization": f'Bearer {access_token}'}

params = {
    "unit": "Minute",
    "interval": "5",
    "barsback": "10"
}

response = requests.request("GET", url, headers=headers, params=params, stream=True)

#print(response.text)
for line in response.iter_lines():
    if line:
        print(line)

response_data: {'access_token': 'eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCIsImtpZCI6Ik56WXpOekExUXpORVJFTXpNMFZHTkVSRVFqSTBSakV4TmpZek56aEdRVFJETmtJd1JVVXpNUSJ9.eyJodHRwOi8vdHJhZGVzdGF0aW9uLmNvbS9pZ25vcmVfZHVhbF9sb2dvbiI6ImZhbHNlIiwiaHR0cDovL3RyYWRlc3RhdGlvbi5jb20vY2xpZW50X3RhZyI6IlEyaXQ4IiwiaHR0cDovL3RyYWRlc3RhdGlvbi5jb20vdXNlcm5hbWUiOiJrbXVycmF5MzAiLCJodHRwOi8vdHJhZGVzdGF0aW9uLmNvbS9mZGNuX2lkIjoiMTI1MTU3ODQiLCJodHRwOi8vdHJhZGVzdGF0aW9uLmNvbS9vbnl4X2lkIjo0NzI1MTMxLCJodHRwOi8vdHJhZGVzdGF0aW9uLmNvbS90ZW1wX2NyZWRlbnRpYWxzIjpmYWxzZSwiaXNzIjoiaHR0cHM6Ly9zaWduaW4udHJhZGVzdGF0aW9uLmNvbS8iLCJzdWIiOiJhdXRoMHwxMjUxNTc4NCIsImF1ZCI6WyJodHRwczovL2FwaS50cmFkZXN0YXRpb24uY29tIiwiaHR0cHM6Ly90cmFkZXN0YXRpb24tcHJvZC50c2xvZ2luLmF1dGgwLmNvbS91c2VyaW5mbyJdLCJpYXQiOjE3Njc2MTkwMzMsImV4cCI6MTc2NzYyMDIzMywic2NvcGUiOiJvcGVuaWQgcHJvZmlsZSBNYXJrZXREYXRhIFJlYWRBY2NvdW50IFRyYWRlIE1hdHJpeCBPcHRpb25TcHJlYWRzIG9mZmxpbmVfYWNjZXNzIiwiYXpwIjoiN3NqQWpsZVVjeFJpWWpwZjRNOUxCRjN1RGJONFV1bXgifQ.lzjdcvwc_0fg1EWNSAlOTFWfa8IYbpHOuKeLlLF

KeyboardInterrupt: 

In [9]:
# get symbol details
access_token = get_access_token()

url = f"{core_url}/v3/marketdata/symbols/ES"

headers = {"Authorization": f'Bearer {access_token}'}

response = requests.request("GET", url, headers=headers)

json_data = response.json()
print(json.dumps(json_data, indent=4, sort_keys=False))

response_data: {'access_token': 'eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCIsImtpZCI6Ik56WXpOekExUXpORVJFTXpNMFZHTkVSRVFqSTBSakV4TmpZek56aEdRVFJETmtJd1JVVXpNUSJ9.eyJodHRwOi8vdHJhZGVzdGF0aW9uLmNvbS9pZ25vcmVfZHVhbF9sb2dvbiI6ImZhbHNlIiwiaHR0cDovL3RyYWRlc3RhdGlvbi5jb20vY2xpZW50X3RhZyI6IlEyaXQ4IiwiaHR0cDovL3RyYWRlc3RhdGlvbi5jb20vdXNlcm5hbWUiOiJrbXVycmF5MzAiLCJodHRwOi8vdHJhZGVzdGF0aW9uLmNvbS9mZGNuX2lkIjoiMTI1MTU3ODQiLCJodHRwOi8vdHJhZGVzdGF0aW9uLmNvbS9vbnl4X2lkIjo0NzI1MTMxLCJodHRwOi8vdHJhZGVzdGF0aW9uLmNvbS90ZW1wX2NyZWRlbnRpYWxzIjpmYWxzZSwiaXNzIjoiaHR0cHM6Ly9zaWduaW4udHJhZGVzdGF0aW9uLmNvbS8iLCJzdWIiOiJhdXRoMHwxMjUxNTc4NCIsImF1ZCI6WyJodHRwczovL2FwaS50cmFkZXN0YXRpb24uY29tIiwiaHR0cHM6Ly90cmFkZXN0YXRpb24tcHJvZC50c2xvZ2luLmF1dGgwLmNvbS91c2VyaW5mbyJdLCJpYXQiOjE3NjgxNjYwMTksImV4cCI6MTc2ODE2NzIxOSwic2NvcGUiOiJvcGVuaWQgcHJvZmlsZSBNYXJrZXREYXRhIFJlYWRBY2NvdW50IFRyYWRlIE1hdHJpeCBPcHRpb25TcHJlYWRzIG9mZmxpbmVfYWNjZXNzIiwiYXpwIjoiN3NqQWpsZVVjeFJpWWpwZjRNOUxCRjN1RGJONFV1bXgifQ.D5I8R-bQYnlWyAq6xVCmRMCmyN-ucB-OjW5PcLr

In [10]:
# get accounts

access_token = get_access_token() # get a new access token
url = f"{core_url}/v3/brokerage/accounts"

headers = {'Authorization': f'Bearer {access_token}' }

response = requests.request("GET", url, headers=headers)
json_data = response.json()
print(json.dumps(json_data, indent=4, sort_keys=False))

response_data: {'access_token': 'eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCIsImtpZCI6Ik56WXpOekExUXpORVJFTXpNMFZHTkVSRVFqSTBSakV4TmpZek56aEdRVFJETmtJd1JVVXpNUSJ9.eyJodHRwOi8vdHJhZGVzdGF0aW9uLmNvbS9pZ25vcmVfZHVhbF9sb2dvbiI6ImZhbHNlIiwiaHR0cDovL3RyYWRlc3RhdGlvbi5jb20vY2xpZW50X3RhZyI6IlEyaXQ4IiwiaHR0cDovL3RyYWRlc3RhdGlvbi5jb20vdXNlcm5hbWUiOiJrbXVycmF5MzAiLCJodHRwOi8vdHJhZGVzdGF0aW9uLmNvbS9mZGNuX2lkIjoiMTI1MTU3ODQiLCJodHRwOi8vdHJhZGVzdGF0aW9uLmNvbS9vbnl4X2lkIjo0NzI1MTMxLCJodHRwOi8vdHJhZGVzdGF0aW9uLmNvbS90ZW1wX2NyZWRlbnRpYWxzIjpmYWxzZSwiaXNzIjoiaHR0cHM6Ly9zaWduaW4udHJhZGVzdGF0aW9uLmNvbS8iLCJzdWIiOiJhdXRoMHwxMjUxNTc4NCIsImF1ZCI6WyJodHRwczovL2FwaS50cmFkZXN0YXRpb24uY29tIiwiaHR0cHM6Ly90cmFkZXN0YXRpb24tcHJvZC50c2xvZ2luLmF1dGgwLmNvbS91c2VyaW5mbyJdLCJpYXQiOjE3NjgxNjYwMjMsImV4cCI6MTc2ODE2NzIyMywic2NvcGUiOiJvcGVuaWQgcHJvZmlsZSBNYXJrZXREYXRhIFJlYWRBY2NvdW50IFRyYWRlIE1hdHJpeCBPcHRpb25TcHJlYWRzIG9mZmxpbmVfYWNjZXNzIiwiYXpwIjoiN3NqQWpsZVVjeFJpWWpwZjRNOUxCRjN1RGJONFV1bXgifQ.jjFKNbFyx6Md0pRlZQHe5QYWQEeVIHLlJ0cOZ06

In [62]:
account_id = os.environ.get('ACCOUNT_ID')
print(f"Account ID: {account_id}")

Account ID: SIM2977785M


In [63]:
# get balances real time
access_token = get_access_token()

url = f"{core_url}/v3/brokerage/accounts/{account_id}/balances"

headers = {"Authorization": f'Bearer {access_token}'}

response = requests.request("GET", url, headers=headers)

print(response.text)

response_data: {'access_token': 'eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCIsImtpZCI6Ik56WXpOekExUXpORVJFTXpNMFZHTkVSRVFqSTBSakV4TmpZek56aEdRVFJETmtJd1JVVXpNUSJ9.eyJodHRwOi8vdHJhZGVzdGF0aW9uLmNvbS9pZ25vcmVfZHVhbF9sb2dvbiI6ImZhbHNlIiwiaHR0cDovL3RyYWRlc3RhdGlvbi5jb20vY2xpZW50X3RhZyI6IlEyaXQ4IiwiaHR0cDovL3RyYWRlc3RhdGlvbi5jb20vdXNlcm5hbWUiOiJrbXVycmF5MzAiLCJodHRwOi8vdHJhZGVzdGF0aW9uLmNvbS9mZGNuX2lkIjoiMTI1MTU3ODQiLCJodHRwOi8vdHJhZGVzdGF0aW9uLmNvbS9vbnl4X2lkIjo0NzI1MTMxLCJodHRwOi8vdHJhZGVzdGF0aW9uLmNvbS90ZW1wX2NyZWRlbnRpYWxzIjpmYWxzZSwiaXNzIjoiaHR0cHM6Ly9zaWduaW4udHJhZGVzdGF0aW9uLmNvbS8iLCJzdWIiOiJhdXRoMHwxMjUxNTc4NCIsImF1ZCI6WyJodHRwczovL2FwaS50cmFkZXN0YXRpb24uY29tIiwiaHR0cHM6Ly90cmFkZXN0YXRpb24tcHJvZC50c2xvZ2luLmF1dGgwLmNvbS91c2VyaW5mbyJdLCJpYXQiOjE3Njc2MTk5NDQsImV4cCI6MTc2NzYyMTE0NCwic2NvcGUiOiJvcGVuaWQgcHJvZmlsZSBNYXJrZXREYXRhIFJlYWRBY2NvdW50IFRyYWRlIE1hdHJpeCBPcHRpb25TcHJlYWRzIG9mZmxpbmVfYWNjZXNzIiwiYXpwIjoiN3NqQWpsZVVjeFJpWWpwZjRNOUxCRjN1RGJONFV1bXgifQ.DUNW5IM8eLtAL9u1T43fE31ShMM9o9IR4lqOdbf

In [ ]:
# get balance beginning of day
access_token = get_access_token()

url = f"{core_url}/v3/brokerage/accounts/{account_id}/bodbalances"

headers = {"Authorization": f'Bearer {access_token}'}

response = requests.request("GET", url, headers=headers)

print(response.text)

In [ ]:
# get historical orders
access_token = get_access_token()

url = f"{core_url}/v3/brokerage/accounts/{account_id}/historicalorders"

querystring = {"since":"2024-09-01"}

headers = {"Authorization": f'Bearer {access_token}'}

response = requests.request("GET", url, headers=headers, params=querystring)

print(response.text)

In [ ]:
# get current orders - today's orders + active orders
access_token = get_access_token()

url = f"{core_url}/v3/brokerage/accounts/{account_id}/orders"

headers = {"Authorization": f'Bearer {access_token}'}

response = requests.request("GET", url, headers=headers)
json_data = response.json()
print(json.dumps(json_data, indent=4, sort_keys=False))

In [ ]:
# stream orders
access_token = get_access_token()

url = f"{core_url}/v3/brokerage/stream/accounts/{account_id}/orders"

headers = {"Authorization": f'Bearer {access_token}'}

response = requests.request("GET", url, headers=headers, stream=True)

for line in response.iter_lines():
    if line:
        print(line)

In [ ]:
# get positions
access_token = get_access_token()

url = f"{core_url}/v3/brokerage/accounts/{account_id}/positions"

headers = {"Authorization": f'Bearer {access_token}'}

response = requests.request("GET", url, headers=headers)

print(response.text)

In [ ]:
# stream positions
access_token = get_access_token()

url = f"{core_url}/v3/brokerage/stream/accounts/{account_id}/positions"

headers = {"Authorization": f'Bearer {access_token}'}

response = requests.request("GET", url, headers=headers, stream=True)

for line in response.iter_lines():
    if line:
        print(line)

In [ ]:
# confirm order
access_token = get_access_token()

url = f"{core_url}/v3/orderexecution/orderconfirm"

payload = {
    "AccountID": account_id,
    "Symbol": "ESZ24",
    "Quantity": "1",
    "OrderType": "Market",
    "TradeAction": "SELL",
    "TimeInForce": {"Duration": "DAY"},
    "Route": "Intelligent"
}
headers = {
    "content-type": "application/json",
    "Authorization": f'Bearer {access_token}'
}

response = requests.request("POST", url, json=payload, headers=headers)

#print(response.text)
json_data = response.json()
print(json.dumps(json_data, indent=4, sort_keys=False))

In [ ]:
# place order
access_token = get_access_token()

url = f"{core_url}/v3/orderexecution/orders"

payload = {
    "AccountID": account_id,
    "Symbol": "ESZ24",
    "Quantity": "1",
    "OrderType": "Market",
    "TradeAction": "SELL",
    "TimeInForce": {"Duration": "DAY"},
    "Route": "Intelligent"
}
headers = {
    "content-type": "application/json",
    "Authorization": f'Bearer {access_token}'
}

response = requests.request("POST", url, json=payload, headers=headers)

print(response.text)

In [ ]:
# cancel order
access_token = get_access_token()

order_id = 
url = f"{core_url}/v3/orderexecution/orders/{order_id}"

headers = {"Authorization": f'Bearer {access_token}'}

response = requests.request("DELETE", url, headers=headers)

print(response.text)

In [ ]:
# get routes
access_token = get_access_token()

url = f"{core_url}/v3/orderexecution/routes"

headers = {"Authorization": f'Bearer {access_token}'}

response = requests.request("GET", url, headers=headers)


json_data = response.json()
print(json.dumps(json_data, indent=4, sort_keys=False))

In [ ]:
# stream tick bars
access_token = get_access_token()

# {symbol}/{interval}/{barsBack}
url = f"{core_url}/v2/stream/tickbars/@ES/100/5"

headers = {"Authorization": f'Bearer {access_token}'}

response = requests.request("GET", url, headers=headers, stream=True)

for line in response.iter_lines():
    if line:
        print(line)

In [ ]:
# automated contract roll for futures
symbol = "ESU24" 
position = 1

access_token = get_access_token()

url = f"{core_url}/v3/marketdata/symbols/@ES"

headers = {"Authorization": f'Bearer {access_token}'}

response = requests.request("GET", url, headers=headers)

json_data = response.json()

top_month_contract = json_data['Symbols'][0]['Underlying']
print(top_month_contract)

# if the contract we're trading does not match the top month
if symbol != top_month_contract:
    # if we have a position
    if position != 0:
        # exit the position in the old contract
        placeOrder(symbol, -1 * position)
        # update our contract to the top month
        symbol = top_month_contract
        # re-open our position in the new contract
        placeOrder(symbol, position)
    else:
        # update our contract to the top month
        symbol = top_month_contract